[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C40_Research_Methodology_Course/03_statistics_for_research/03_statistics_for_research.ipynb)

# 03 · 研究统计（动手做）

目标：把统计严谨性跑成代码——**多 seed 方差**、**bootstrap CI**、**功效/样本量**、**多重比较校正(Bonferroni/Holm)**、**效应量(Cohen's d)**。

路线：种子方差 → 从零 bootstrap CI → 功效分析估种子数 → 多重比较假阳性 → Bonferroni/Holm → 效应量 → ✏️ 练习 → 📖 答案 → 🧪 真实方差数字胶囊。

> 核心心法：**一个没有方差的数字等于没有信息；显著 ≠ 重要。**

## 1 · worked：随机种子方差能淹没真实差异

真实差异 +0.005，但种子方差 ±0.01。看单次运行如何被噪声主导，多次运行如何让信号浮现。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

TRUE_A, TRUE_B = 0.880, 0.885    # B 真实只高 0.005
NOISE = 0.010                    # 种子方差比真实差异还大！

def runs(true, n, rng):
    return true + rng.normal(0, NOISE, size=n)

# 单次：B 赢的比例（真实差异被噪声淹没 -> 接近抛硬币）
single = sum(runs(TRUE_B,1,rng)[0] > runs(TRUE_A,1,rng)[0] for _ in range(3000)) / 3000
print(f'真实差异仅 +0.005，但噪声 ±0.010')
print(f'单次运行 B 赢的比例 = {single:.0%}  (噪声>信号 -> 接近抛硬币)')
assert 0.5 < single < 0.75, '信号略大于噪声时略高于50%但远非确定'

# 多次：跑 30 个种子取均值
a30, b30 = runs(TRUE_A, 30, rng), runs(TRUE_B, 30, rng)
print(f'\n跑30种子: A={a30.mean():.4f}±{a30.std(ddof=1):.4f}  B={b30.mean():.4f}±{b30.std(ddof=1):.4f}')
print(f'均值差 = {b30.mean()-a30.mean():+.4f}  (接近真实 +0.005)')
assert abs((b30.mean()-a30.mean()) - 0.005) < 0.006, '多种子均值差应接近真实差异'
print('✅ 单次几乎看不出，多种子才让 +0.005 的真实差异浮现。')

## 2 · worked：从零实现 bootstrap 置信区间

Bootstrap：把样本当总体，有放回重采样 B 次、各算统计量、取 2.5%/97.5% 分位数 = 95% CI。不需要分布假设。

In [ ]:
def bootstrap_ci(data, statistic=np.mean, B=10000, alpha=0.05, seed=0):
    '''返回 statistic 的 (1-alpha) percentile bootstrap 置信区间 (lo, hi)。'''
    rng = np.random.default_rng(seed)
    data = np.asarray(data)
    n = len(data)
    stats = np.empty(B)
    for b in range(B):
        sample = data[rng.integers(0, n, size=n)]   # 有放回重采样 n 个
        stats[b] = statistic(sample)
    lo = np.percentile(stats, 100 * alpha / 2)
    hi = np.percentile(stats, 100 * (1 - alpha / 2))
    return lo, hi

data = np.array([88.1, 90.2, 89.5, 90.4, 89.0, 88.7, 89.9, 90.1])
lo, hi = bootstrap_ci(data, np.mean)
print(f'样本均值 = {data.mean():.3f}')
print(f'bootstrap 95% CI = [{lo:.3f}, {hi:.3f}]')
assert lo < data.mean() < hi, '均值应落在自己的CI内'
assert hi - lo > 0, 'CI 应有正宽度'
# bootstrap 也能用于中位数等任意统计量
mlo, mhi = bootstrap_ci(data, np.median)
print(f'中位数的 95% CI = [{mlo:.3f}, {mhi:.3f}]  <- 同一套方法，换个统计量即可')
print('✅ bootstrap 对任意统计量给出置信区间，无需分布假设。')

## 3 · worked：bootstrap 差异的 CI 是否跨过 0

判断「B 比 A 好」是否可信：对**差异**做 bootstrap，看 95% CI 是否包含 0。不含 0 → 差异可信；含 0 → 可能是噪声。
顺便看：同一个真实差异，3 个种子 vs 30 个种子，结论可能相反。

In [ ]:
def diff_ci(a, b, B=10000, seed=0):
    '''对 mean(b)-mean(a) 做 bootstrap，返回差异的 95% CI。'''
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    diffs = np.empty(B)
    for i in range(B):
        sa = a[rng.integers(0, len(a), len(a))]
        sb = b[rng.integers(0, len(b), len(b))]
        diffs[i] = sb.mean() - sa.mean()
    return np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

rng = np.random.default_rng(7)
GAP = 0.5                         # B 真实高 0.5 分
for n in [3, 30]:
    a = 88.0 + rng.normal(0, 0.4, n)
    b = 88.0 + GAP + rng.normal(0, 0.4, n)
    lo, hi = diff_ci(a, b)
    crosses_zero = lo <= 0 <= hi
    verdict = '不显著(CI跨0)' if crosses_zero else '显著(CI不含0)'
    print(f'n={n:2d}: 差异CI=[{lo:+.3f},{hi:+.3f}]  -> {verdict}')
print('✅ 同样真实的 +0.5：种子少时 CI 跨0(看不出)，种子多时 CI 不含0(看得出)。')
print('   样本量决定了你能否看见真实存在的效应 —— 这就是『功效』。')

## 4 · worked：功效分析——该跑多少个种子？

给定想检测的效应量、方差、目标功效(0.8)、显著水平(0.05)，用 Monte Carlo 估计**所需种子数**：
对每个候选 n，模拟很多次实验，看有多少比例能检测到差异(功效)。

In [ ]:
def estimate_power(n, true_gap, noise, n_sims=2000, alpha=0.05, seed=0):
    '''模拟 n 个种子的实验 n_sims 次，返回功效 = 检测到差异的比例。
       检测准则：差异的 bootstrap-free 近似——用两样本均值差 / 合并SE 的 |z|>1.96。'''
    rng = np.random.default_rng(seed)
    detected = 0
    for _ in range(n_sims):
        a = rng.normal(0.0,        noise, n)
        b = rng.normal(true_gap,   noise, n)
        se = np.sqrt(a.var(ddof=1)/n + b.var(ddof=1)/n)
        z = (b.mean() - a.mean()) / se if se > 0 else 0
        if abs(z) > 1.96:                # 近似 alpha=0.05 双侧
            detected += 1
    return detected / n_sims

print('检测真实差异 +0.5(噪声±1.0)所需的种子数：')
print(f"{'种子数 n':>8s} {'功效':>8s}")
needed = None
for n in [5, 10, 20, 40, 80]:
    pw = estimate_power(n, true_gap=0.5, noise=1.0)
    print(f'{n:>8d} {pw:>7.0%}')
    if needed is None and pw >= 0.80:
        needed = n
print(f'\n达到 80% 功效约需 n >= {needed} 个种子')
assert estimate_power(80,0.5,1.0) > estimate_power(5,0.5,1.0), '种子越多功效越高'
assert needed is not None and needed >= 20, '小效应+大噪声需要不少种子'
print('✅ 功效分析让你在跑之前就知道：要检测这个效应，得跑多少种子。')

## 5 · worked：多重比较——比较越多假阳性越多 + Holm 校正

20 个『其实一样好』的方法两两/各自比一次，即使无真效应，也几乎必出『显著』赢家。再看 Bonferroni/Holm 如何控制。

In [ ]:
# 演示：m 个方法真实水平全相同，各做一次检验，统计『至少一个假阳性』
def family_false_positive_rate(m, n_families=3000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    any_fp = 0
    for _ in range(n_families):
        # m 个 p 值（无真效应时 p 服从均匀分布）
        pvals = rng.uniform(0, 1, size=m)
        if (pvals < alpha).any():       # 未校正：任一 p<0.05 即报假阳性
            any_fp += 1
    return any_fp / n_families

for m in [1, 10, 20]:
    fpr = family_false_positive_rate(m)
    theory = 1 - (1 - 0.05) ** m
    print(f'm={m:2d} 个比较: 至少一个假阳性 实测={fpr:.0%} 理论={theory:.0%}')
assert family_false_positive_rate(20) > 0.5, '20个比较假阳性率应>50%'

def holm_correction(pvals, alpha=0.05):
    '''Holm-Bonferroni：返回每个 p 值是否被判显著(布尔数组)。'''
    pvals = np.asarray(pvals)
    m = len(pvals)
    order = np.argsort(pvals)            # 升序
    reject = np.zeros(m, dtype=bool)
    for rank, idx in enumerate(order):
        thresh = alpha / (m - rank)      # 第 rank 小用 alpha/(m-rank)
        if pvals[idx] <= thresh:
            reject[idx] = True
        else:
            break                        # 一旦不通过就停止
    return reject

pvals = [0.001, 0.012, 0.03, 0.04, 0.20]    # 5 个比较
raw_sig = np.array(pvals) < 0.05
bonf_sig = np.array(pvals) < 0.05 / len(pvals)
holm_sig = holm_correction(pvals)
print(f'\n未校正显著: {raw_sig.sum()} 个   Bonferroni: {bonf_sig.sum()} 个   Holm: {holm_sig.sum()} 个')
assert holm_sig.sum() >= bonf_sig.sum(), 'Holm 功效 >= Bonferroni（发现的真效应不更少）'
assert raw_sig.sum() >= holm_sig.sum(), '校正后显著数不会变多'
print('✅ 多比较必须校正；Holm 比 Bonferroni 功效更高，应优先。')

## 6 · worked：效应量——显著但不重要

大样本能让微不足道的差异也『显著』。算 Cohen's d 看差异到底大不大。

In [ ]:
def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    n1, n2 = len(a), len(b)
    s_pooled = np.sqrt(((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1)) / (n1+n2-2))
    return (b.mean() - a.mean()) / s_pooled

rng = np.random.default_rng(3)
# 巨大样本 + 极小真实差异 -> 统计显著但效应量微不足道
big_a = rng.normal(0.900, 0.02, 100000)
big_b = rng.normal(0.9003, 0.02, 100000)   # 只高 0.0003
se = np.sqrt(big_a.var(ddof=1)/len(big_a) + big_b.var(ddof=1)/len(big_b))
z = (big_b.mean() - big_a.mean()) / se
d = cohens_d(big_a, big_b)
print(f'差异 = {big_b.mean()-big_a.mean():+.5f}')
print(f'|z| = {abs(z):.1f}  (远超1.96 -> 统计极显著)')
print(f"Cohen's d = {d:.3f}  (远小于0.2 -> 效应微不足道)")
assert abs(z) > 1.96, '大样本下微小差异也显著'
assert abs(d) < 0.2, '但效应量极小(无实际意义)'
print('✅ 显著 ≠ 重要：必须同时报效应量，否则被大样本的『显著』误导。')

---
## ✏️ 练习 1：标准误与所需样本量

标准误 SE = σ/√n。要把 SE 降到目标值 `target_se`，需要多少样本？
实现 `n_for_se(sigma, target_se)`：返回满足 σ/√n ≤ target_se 的**最小整数** n。

In [ ]:
def n_for_se(sigma, target_se):
    # TODO: 解 σ/√n <= target_se  =>  n >= (σ/target_se)**2，向上取整
    #   用 import math; math.ceil(...)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
import math
assert n_for_se(1.0, 0.5) == 4,  'σ=1, 要SE<=0.5 -> n>=4'
assert n_for_se(1.0, 0.25) == 16, 'SE 减半 -> n 翻4倍'
assert n_for_se(2.0, 0.5) == 16
print('✅ 练习 1 通过：能算达到目标精度所需的样本量（SE∝1/√n）')

## ✏️ 练习 2：Bonferroni 校正

实现 `bonferroni(pvals, alpha=0.05)`：返回布尔数组，标记每个 p 值在 Bonferroni 校正后是否显著（p < α/m）。

In [ ]:
def bonferroni(pvals, alpha=0.05):
    # TODO: m = len(pvals); 返回 np.array(pvals) < alpha/m
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
p = [0.001, 0.02, 0.04, 0.3]
sig = bonferroni(p)              # 阈值 = 0.05/4 = 0.0125
assert sig.tolist() == [True, False, False, False], '只有 0.001<0.0125'
assert bonferroni([0.04], 0.05).tolist() == [True], 'm=1 时退化为普通阈值'
print('✅ 练习 2 通过：Bonferroni 校正正确')

## ✏️ 练习 3：方差分解——哪个随机源主导？

你有不同(种子, 数据划分)组合的准确率。实现 `variance_breakdown(df)`：
返回 dict，分别给出『固定划分、只变种子』的平均方差，和『固定种子、只变划分』的平均方差，看哪个更大。

In [ ]:
import pandas as pd
def variance_breakdown(df):
    # df 有列: seed, split, acc
    # TODO: 
    #   seed_var  = 对每个 split 内部算 acc 的方差(ddof=1)，再对各 split 取平均
    #   split_var = 对每个 seed 内部算 acc 的方差(ddof=1)，再对各 seed 取平均
    #   返回 dict(seed_var=..., split_var=...)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng = np.random.default_rng(0)
rows = []
for seed in range(4):
    for split in range(4):
        # 让『划分』带来大方差(±0.05)，『种子』带来小方差(±0.005)
        acc = 0.85 + split*0.0 + rng.normal(0,0.005) + (split-1.5)*0.03
        rows.append(dict(seed=seed, split=split, acc=acc))
df = pd.DataFrame(rows)
vb = variance_breakdown(df)
print(f"固定划分变种子的方差 = {vb['seed_var']:.6f}")
print(f"固定种子变划分的方差 = {vb['split_var']:.6f}")
assert vb['split_var'] > vb['seed_var'], '本例中数据划分是更大的方差来源'
print('✅ 练习 3 通过：方差分解揭示『划分』比『种子』贡献更多方差 -> 只控种子不够')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
import math
def n_for_se(sigma, target_se):
    return math.ceil((sigma / target_se) ** 2)

In [ ]:
# 练习 2 参考答案
def bonferroni(pvals, alpha=0.05):
    m = len(pvals)
    return np.asarray(pvals) < alpha / m

In [ ]:
# 练习 3 参考答案
def variance_breakdown(df):
    seed_var = df.groupby('split')['acc'].var(ddof=1).mean()
    split_var = df.groupby('seed')['acc'].var(ddof=1).mean()
    return dict(seed_var=float(seed_var), split_var=float(split_var))

---
## 🧪 真实数据胶囊：复现『只跑一个种子会得出错误排名』(Henderson/Bouthillier 式)

Henderson et al. 与 Bouthillier et al. 的核心实证：在高方差下，**单种子的方法排名极不稳定**——换一批种子，赢家可能换人。
我们用真实文献报告的**典型方差量级**(深度学习准确率运行间 std≈0.5~1.0 分)复现这个现象。

In [ ]:
# 真实场景的典型数字：两个方法，真实差异很小，运行间方差不小
TRUE_GAP = 0.3       # B 真实只比 A 高 0.3 分
STD = 0.8            # 文献报告的典型运行间标准差(~0.5-1.0)

def single_seed_winner(rng):
    '''各跑1个种子，返回谁赢("B" 或 "A")。'''
    a = 80.0 + rng.normal(0, STD)
    b = 80.0 + TRUE_GAP + rng.normal(0, STD)
    return 'B' if b > a else 'A'

rng = np.random.default_rng(2024)
winners = [single_seed_winner(rng) for _ in range(5000)]
b_win_rate = winners.count('B') / len(winners)
print(f'真相：B 比 A 高 {TRUE_GAP} 分(std={STD})')
print(f'但只跑1个种子时，B 被判赢的比例仅 = {b_win_rate:.0%}')
print(f'-> 约 {1-b_win_rate:.0%} 的『研究者』会得出错误结论(A赢)！')
assert 0.55 < b_win_rate < 0.7, '小效应+大方差时，单种子排名近乎掷骰子'
print('✅ 复现 Henderson/Bouthillier 的警告：单种子排名极不可靠，必须多种子+统计。')

**🧪 胶囊练习**：实现 `seeds_for_reliable_ranking(true_gap, std, target=0.95)`：
用 `estimate_power` 找出『以 ≥ target 的功效检测到差异』所需的最小种子数（在 [5,10,20,40,80,160,320] 里找）。

In [ ]:
def seeds_for_reliable_ranking(true_gap, std, target=0.95):
    # TODO: 对 candidates=[5,10,20,40,80,160,320]，用 estimate_power(n, true_gap, std)
    #   返回第一个使功效 >= target 的 n；都不够则返回 None
    raise NotImplementedError

In [ ]:
# 自测
need = seeds_for_reliable_ranking(0.5, 0.8, target=0.95)   # 效应0.5、噪声0.8
print(f'要以 95% 功效检测到 +0.5(噪声0.8)的差异，约需 {need} 个种子')
assert need is not None and need >= 40, '小效应+大方差需要相当多的种子才可靠'
print('✅ 胶囊练习通过：能算出可靠检测差异所需的种子数')

In [ ]:
# 📖 胶囊参考答案
def seeds_for_reliable_ranking(true_gap, std, target=0.95):
    for n in [5, 10, 20, 40, 80, 160, 320]:
        if estimate_power(n, true_gap, std) >= target:
            return n
    return None

### 小结
- **随机种子方差**常大到淹没真实差异；只跑一个种子≈抛硬币定胜负。
- **报均值必带方差**；σ(单次波动)不随n变，SE=σ/√n(均值精度)随n变小。
- **bootstrap**：有放回重采样估任意统计量的 CI，无需分布假设；差异 CI 跨0 = 不显著。
- **功效分析**：跑前算『要多少种子才够』；功效不足时『没差异』无信息。
- **多重比较**必校正(Holm 优于 Bonferroni)；**效应量**(Cohen's d)回答『多大』——显著≠重要。

下一站：**模块 04 · 研究工程卫生** —— 实验多了，怎么管到可复现、可聚合。